# 02 — Chỉ nhập URL thuộc dataset BTC mới (CPU)

**Chọn CPU → Run all.** Dùng một phiên cho notebook này. Luồng submission
vẫn chạy riêng với candidate hiện tại; notebook này không đổi active
candidate, snapshot, model cache hoặc khóa code của luồng đó.

Lượt mặc định `btc-new-71be0e12-20261019-v1`: chỉ sáu frontier mới
00000–00005, gắn với dataset BTC revision `71be0e12`. Không nhập shard
thuộc snapshot cũ. Chọn đúng phiên bản tích lũy
đã audit, kể cả TAR lồng TAR. Đọc nguồn qua Drive API, không cần tạo
shortcut AI Guru hay tải toàn bộ TAR về /content.

Checkpoint và kết quả: `VietMedBridge/data_staging/btc-new-71be0e12-20261019-v1/`.
Ngắt phiên: mở lại cùng notebook, cùng folder, Run all; từng source/shard
hoàn tất được giữ. Shard đang ghi dở sẽ chạy lại. Đây là import, kiểm tra
và chia chunk, chưa chạy embedding hay ghép vào submission.

Ghi Parquet theo nhóm 2.048 dòng; SQL dùng một luồng, spill ở ổ local,
tối đa 2 GiB và không quá 1/4 RAM khả dụng tại lúc bắt đầu xử lý.
Giữ nguyên campaign/code lock khi chạy tiếp sau lỗi thiếu bộ nhớ.


In [ ]:
#@title 1. Kết nối Drive và cài code riêng cho lượt nhập mới
import importlib
import json
import os
import subprocess
import sys
from pathlib import Path

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("Dùng runtime Colab Python 3.11–3.13.")
print("Bắt đầu CPU intake; không nạp model embedding.", flush=True)
from google.colab import drive
drive.mount("/content/drive")

# Chỉ sửa nếu project của team ở vị trí khác; tự tìm shortcut dùng chung.
PROJECT = Path("/content/drive/MyDrive/VietMedBridge")
if not (PROJECT / "data").is_dir():
    shared = Path("/content/drive/.shortcut-targets-by-id/1uKKHYEcMPodnJIscKPi6jsoNBJvyhFGK/VietMedBridge")
    if (shared / "data").is_dir():
        PROJECT = shared
    else:
        raise FileNotFoundError("Chưa tìm thấy folder VietMedBridge dùng chung. Thêm shortcut VietMedBridge vào My Drive.")
CAMPAIGN = "btc-new-71be0e12-20261019-v1"
INTAKE_ROOT = PROJECT.resolve() / "data_staging" / CAMPAIGN
INTAKE_ROOT.mkdir(parents=True, exist_ok=True)
WORK_DIR = Path("/content/vmb_intake_work")
WORK_DIR.mkdir(parents=True, exist_ok=True)
CHECKOUT = Path("/content/VietMedBridge-intake")
REPO_URL = "https://github.com/Platypus27-coder/VietMedBridge.git"
lock_path = INTAKE_ROOT / "intake_code_lock.json"
lock = json.loads(lock_path.read_text()) if lock_path.is_file() else {}
if lock and lock.get("repo_url") != REPO_URL:
    raise ValueError("Intake code lock belongs to another repository.")
if not (CHECKOUT / ".git").is_dir():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(CHECKOUT)], check=True)
remote = subprocess.check_output(["git", "-C", str(CHECKOUT), "remote", "get-url", "origin"], text=True).strip()
if remote != REPO_URL:
    raise ValueError("Checkout intake thuộc repo khác.")
reference = lock.get("commit") or "main"
print("Tải code intake:", reference, flush=True)
subprocess.run(["git", "-C", str(CHECKOUT), "fetch", "--depth", "1", "origin", reference], check=True)
commit = subprocess.check_output(["git", "-C", str(CHECKOUT), "rev-parse", "FETCH_HEAD"], text=True).strip()
if "vietmedbridge" in sys.modules and globals().get("_INTAKE_COMMIT") != commit:
    raise RuntimeError("Restart session để nạp đúng code intake; checkpoint trên Drive vẫn giữ.")
subprocess.run(["git", "-C", str(CHECKOUT), "checkout", "--detach", commit], check=True)
print("Cài dependencies CPU…", flush=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(CHECKOUT)], check=True)
runtime_path = INTAKE_ROOT / "intake_runtime_lock.json"
if runtime_path.is_file():
    pinned = json.loads(runtime_path.read_text())
    packages = [name+"=="+version for name, version in pinned.items() if name != "vietmedbridge"]
    subprocess.run([sys.executable, "-m", "pip", "install", *packages], check=True)
sys.path.insert(0, str(CHECKOUT / "src"))
importlib.invalidate_caches()
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HOME"] = "/content/hf_cache_intake"
from vietmedbridge.artifacts import atomic_json, read_json, runtime_versions
from vietmedbridge.isolated_intake import intake_root
if INTAKE_ROOT != intake_root(PROJECT / "data", CAMPAIGN):
    raise ValueError("Intake output root không tách biệt khỏi submission.")
if not lock:
    atomic_json(lock_path, {"repo_url": REPO_URL, "commit": commit})
if not runtime_path.is_file():
    atomic_json(runtime_path, runtime_versions())
_INTAKE_COMMIT = commit
print("Code:", commit, "\nCheckpoint:", INTAKE_ROOT, flush=True)


In [ ]:
#@title 2. Nhận checkpoint trước; chỉ nạp tài nguyên khi cần xử lý phần thiếu
"""Manifest-first intake controller, embedded into the Colab notebook.

Kept outside src/ deliberately: changing resume orchestration must not change
the pinned producer fingerprint or invalidate existing document/chunk shards.
Completed receipts are NOT a fresh payload-integrity audit: only metadata is
read for status; mapping payloads are SHA-checked when actually used.
"""
from collections import Counter
from contextlib import contextmanager
from dataclasses import asdict
from pathlib import Path
import re
import tempfile

from vietmedbridge import isolated_intake as intake_core
from vietmedbridge.artifacts import (atomic_json, code_fingerprint, digest_json,
    read_json, runtime_versions, sha256_file, utc_now, verify_file)
from vietmedbridge.dataset import _connection


def intake_memory_mb():
    """At most 2 GiB and a quarter of available RAM; no reservation or GPU use."""
    available = []
    try:
        match = re.search(r"^MemAvailable:\s+(\d+) kB", Path("/proc/meminfo").read_text(), re.M)
        if match:
            available.append(int(match[1])*1024)
    except OSError:
        pass
    for limit, used in (("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory.current"),
                        ("/sys/fs/cgroup/memory/memory.limit_in_bytes", "/sys/fs/cgroup/memory/memory.usage_in_bytes")):
        try:
            available.append(max(0, int(Path(limit).read_text())-int(Path(used).read_text())))
        except (OSError, ValueError):
            pass
    return min(2048, max(64, min(available)//(4*1024**2))) if available else 512


def bounded_intake_parquet(con, query, destination, work_dir):
    """Flush wide text every 2,048 rows, then atomically publish the full file."""
    with tempfile.TemporaryDirectory(prefix="intake-table-", dir=work_dir) as temporary:
        path = Path(temporary)/"table.parquet"
        con.sql(query).write_parquet(str(path), compression="zstd", row_group_size=2048)
        return intake_core.publish_file(path, destination)


@contextmanager
def bounded_intake_sql(memory_mb=None):
    """Scoped I/O adapter for the single intake controller, including old locks.

    The pinned producer files, policies and completed selections stay untouched.
    Only this module's selection/outcome SQL and Parquet writer are adapted;
    normal importer/chunk shards still use their original producer.
    """
    budget = intake_memory_mb() if memory_mb is None else memory_mb
    if type(budget) is not int or not 64 <= budget <= 2048:
        raise ValueError("Intake SQL budget must be 64..2048 MiB.")
    original_connection, original_parquet = intake_core._connection, intake_core._parquet

    def connect(temp_dir, memory_limit=None):
        con = _connection(temp_dir, memory_limit=memory_limit or f"{budget}MiB")
        try:
            con.execute("SET threads=1")
            con.execute("SET preserve_insertion_order=false")
        except BaseException:
            con.close()
            raise
        return con

    intake_core._connection, intake_core._parquet = connect, bounded_intake_parquet
    print(f"Intake SQL: RAM limit {budget} MiB | 1 thread | local spill | Parquet groups 2,048 rows", flush=True)
    try:
        yield
    finally:
        intake_core._connection, intake_core._parquet = original_connection, original_parquet


class IntakeResume:
    def __init__(self, plan, *, root, work_dir, tokenizer_spec, chunk_config, shard_size=2048):
        self.plan, self.root, self.work = plan, Path(root), Path(work_dir)
        self.spec, self.config, self.shard_size = tokenizer_spec, chunk_config, shard_size
        self.plan_sha = digest_json(plan)
        if any((p/"active_data_candidate.json").exists() for p in (self.root, *self.root.parents)):
            raise ValueError("Intake must stay outside the active submission data root.")
        if any(not re.fullmatch(r"[a-zA-Z0-9_-]+", s["job_id"]) for s in plan["sources"]):
            raise ValueError("Invalid intake job ID.")
        intake_core.pin_plan(self.root, plan)
        self.work.mkdir(parents=True, exist_ok=True)
        self.contract = {"tokenizer": tokenizer_spec, "chunking": asdict(chunk_config),
            "shard_size": shard_size, "code_sha256": code_fingerprint(), "runtime": runtime_versions()}
        contract = self.root/"intake_contract.json"
        if contract.exists() and read_json(contract) != self.contract:
            raise ValueError("Pinned intake code/runtime/chunk policy changed; do not discard checkpoints.")
        if not contract.exists():
            atomic_json(contract, self.contract)
        self.refresh()

    def _file(self, root, item):
        path = intake_core._inside(root, item["path"])
        stat = path.stat()  # Missing/changed files fail; never silently reprocess a completed source.
        if stat.st_size <= 0 or ("bytes" in item and stat.st_size != item["bytes"]):
            raise ValueError(f"Checkpoint artifact size changed: {path}")
        return path

    def refresh(self):
        self.completed = []
        gap = False
        for source in self.plan["sources"]:
            target = self.root/"sources"/source["job_id"]
            receipt = target/"complete.json"
            if not receipt.exists():
                gap = True
                continue
            if gap:
                raise ValueError("Out-of-order complete receipt; preserve files and inspect source lineage.")
            saved = read_json(receipt)
            if (saved.get("state") != "COMPLETE" or saved["source"] != source
                    or saved["plan_sha256"] != self.plan_sha):
                raise ValueError("Completed source/plan binding changed.")
            for kind, name, sha_key in (("added_ids", "added_ids.parquet", "added_ids_sha256"),
                                       ("outcomes", "source_outcomes.parquet", "outcomes_sha256")):
                item = saved.get("files", {}).get(kind, {"path": name, "sha256": saved[sha_key]})
                if item["sha256"] != saved[sha_key]:
                    raise ValueError("Receipt payload binding changed.")
                self._file(target, item)
            self.completed.append(saved)

    def status(self):
        counts = Counter()
        for saved in self.completed:
            counts.update(saved["counts"])
        return {"state": "INTAKE_COMPLETE" if len(self.completed) == len(self.plan["sources"]) else "INTAKE_IN_PROGRESS",
            "plan_sha256": self.plan_sha, "completed_sources": len(self.completed),
            "total_sources": len(self.plan["sources"]), "counts": dict(counts),
            "builds": [{"build_run": s["build_run"], "snapshot_sha256": s["build_snapshot_sha256"],
                "snapshot_id": s["source"]["snapshot_id"]} for s in self.completed if s["build_run"]],
            "resume_policy": "MANIFEST_FIRST_MAPPING_SHA_CHECKED_ON_USE",
            "scope": "CPU import, quality and exact-span chunks only; not embedded/frozen/promoted or submitted",
            "submission_candidate_changed": False, "updated_at": utc_now()}

    def _selection(self, source, target, existing):
        marker = target/"selection/selection.json"
        if not marker.exists():
            return None
        saved = read_json(marker)
        metadata = read_json(target/"metadata/metadata.json")
        expected = {"metadata": metadata,
            "links_sha256": self.plan["snapshots"][source["snapshot_id"]]["sha256"],
            "existing": [{"sha256": checksum} for _, checksum in existing]}
        if (saved["identity"] != expected or metadata["source"] != source
                or metadata["files"]["frontier"]["sha256"] != source["frontier_sha256"]
                or sum(saved["counts"].values()) != saved["official_ids"]):
            raise ValueError("Saved selection binding changed; no automatic rebuild.")
        for item in saved["files"].values():
            self._file(target/"selection", item)
        print(f"REUSE_SELECTION {source['job_id']} | no archive/baseline rescan", flush=True)
        return saved, metadata

    def _validated_build(self, source, selection, build_dir):
        path = build_dir/"build.json"
        if not path.exists():
            return None
        build = read_json(path)
        if not build.get("selected_range_complete"):
            return None
        if build.get("state") != "DATA_VALIDATED" or not build.get("integrity", {}).get("passed"):
            raise ValueError("Complete build has not passed integrity; retain and inspect it.")
        config = read_json(build_dir/"config.json")
        queued = selection["counts"].get("QUEUED_FOR_PIPELINE", 0)
        files = {kind: {"name": Path(intake_core.MEMBERS[kind]).name,
            "sha256": selection["files"][kind]["sha256"],
            "bytes": self._file(self.root/"sources"/source["job_id"]/"selection", selection["files"][kind]).stat().st_size}
            for kind in intake_core.MEMBERS}
        if (config["signature"] != digest_json({k: v for k, v in config.items() if k != "signature"})
                or config["signature"] != build["signature"]
                or config["source_input_signature"] != digest_json(files)
                or config["external_snapshot_id"] != source["snapshot_id"]
                or config["official_links_sha256"] != self.plan["snapshots"][source["snapshot_id"]]["sha256"]
                or config["tokenizer"] != self.spec or config["chunks"] != asdict(self.config)
                or config["code_sha256"] != self.contract["code_sha256"]
                or config["runtime"] != self.contract["runtime"] or config["shard_size"] != self.shard_size
                or build["requested_input_records"] != queued or build["counts"]["input_records"] != queued
                or build["snapshot_sha256"] != digest_json({"signature": build["signature"], "parts": build["parts"]})):
            raise ValueError("Validated build identity differs from the saved selection/policy.")
        for part in build["parts"]:
            for kind, item in part["files"].items():
                path = self._file(build_dir, item)
                if kind in ("documents", "failures"):
                    # Only these payloads are about to be used to finish the
                    # ID/outcome mapping. Child/parent chunks stay deferred to freeze.
                    verify_file(path, item["sha256"])
        print(f"REUSE_VALIDATED_BUILD {source['job_id']} | no re-chunk or full validation replay", flush=True)
        return build

    @bounded_intake_sql()
    def run(self, *, load_reader, load_links, load_baseline, load_tokenizer):
        self.refresh()
        for saved in self.completed:
            print(f"SKIP_COMPLETED_SOURCE {saved['source']['job_id']} | receipt only; payload checked on use", flush=True)
        if len(self.completed) == len(self.plan["sources"]):
            result = self.status()
            atomic_json(self.root/"intake_status.json", result)
            return result
        # Paths/checksums suffice until a NEW selection really needs a mapping.
        existing = [(None, self.plan["baseline"]["sha256"])] + [
            (self.root/"sources"/s["source"]["job_id"]/"added_ids.parquet", s["added_ids_sha256"])
            for s in self.completed]
        for source in self.plan["sources"][len(self.completed):]:
            job = source["job_id"]
            target, build_dir = self.root/"sources"/job, self.root/"processed"/job
            print(f"CONTINUE_SOURCE {job} | {len(self.completed)}/{len(self.plan['sources'])} sources complete", flush=True)
            cached = self._selection(source, target, existing)
            if cached is None:
                metadata = intake_core.stage_metadata(load_reader(), source, target/"metadata")
                local_maps = [load_baseline() if path is None else intake_core._local_mapping(path, checksum, self.work)
                              for path, checksum in existing]
                for path, (_, checksum) in zip(local_maps, existing):
                    verify_file(path, checksum)
                selection = intake_core.select_delta(target/"metadata", metadata, load_links(source["snapshot_id"]),
                    local_maps, target/"selection", self.work)
            else:
                selection, metadata = cached
            build = None
            if selection["counts"].get("QUEUED_FOR_PIPELINE", 0):
                build = self._validated_build(source, selection, build_dir)
                if build is None:
                    markers = sorted((build_dir/"parts").glob("*.done.json"))
                    committed = [read_json(path) for path in markers]
                    print(f"RESUME_PARTS {job} | {len(committed)} committed shards, "
                          f"{sum(p['records'] for p in committed):,} IDs; verify current source only", flush=True)
                    staged = intake_core.restore_import_source(target/"selection", selection, self.work/"sources"/job)
                    for kind in intake_core.MEMBERS:
                        item = selection["files"][kind]
                        verify_file(staged/item["path"], item["sha256"])
                    tokenizer, actual_spec = load_tokenizer()
                    if actual_spec != self.spec:
                        raise ValueError("Loaded tokenizer differs from the pinned contract.")
                    build = intake_core.import_external_corpus(staged, load_links(source["snapshot_id"]),
                        self.root/"processed", tokenizer, self.spec, run_name=job, config=self.config,
                        work_dir=self.work, shard_size=self.shard_size)
                    if not build["selected_range_complete"] or not build["integrity"]["passed"]:
                        raise ValueError("Intake build is not complete and valid.")
                atomic_json(build_dir/"intake_provenance.json", {"source": source,
                    "original_metadata": metadata["files"], "plan_sha256": self.plan_sha,
                    "raw_locator": "Drive file ID -> optional inner_tar -> suffix after # in document.raw_file",
                    "raw_verification": "METADATA_ONLY_RAW_PAYLOAD_NOT_REPLAYED"})
            entry = selection["files"]["selection"]
            verify_file(target/"selection"/entry["path"], entry["sha256"])
            counts = intake_core._write_outcomes(target, target/"selection", selection, build_dir, build, self.work)
            with _connection(self.work) as con:
                if build:
                    con.read_parquet([str(build_dir/p["files"]["documents"]["path"]) for p in build["parts"]]).create_view("added")
                    query = "SELECT doc_id,url FROM added"
                else:
                    query = "SELECT NULL::BIGINT AS doc_id,NULL::VARCHAR AS url WHERE false"
                intake_core._parquet(con, query, target/"added_ids.parquet", self.work)
            files = {kind: {"path": name, "sha256": sha256_file(target/name), "bytes": (target/name).stat().st_size}
                     for kind, name in (("added_ids", "added_ids.parquet"), ("outcomes", "source_outcomes.parquet"))}
            saved = {"source": source, "plan_sha256": self.plan_sha, "state": "COMPLETE", "counts": counts,
                "build_run": job if build else None, "build_snapshot_sha256": build["snapshot_sha256"] if build else None,
                "data_counts": build["counts"] if build else {}, "files": files,
                "added_ids_sha256": files["added_ids"]["sha256"], "outcomes_sha256": files["outcomes"]["sha256"],
                "updated_at": utc_now()}
            atomic_json(target/"complete.json", saved)  # last: durable commit of the whole source
            self.completed.append(saved)
            existing.append((target/"added_ids.parquet", saved["added_ids_sha256"]))
            result = self.status()
            atomic_json(self.root/"intake_status.json", result)
            print(f"SOURCE_CHECKPOINT_SAVED {job} | {counts}", flush=True)
        return result


import json
from huggingface_hub import hf_hub_download
from vietmedbridge.artifacts import verify_file, publish_file, digest_json
from vietmedbridge.isolated_intake import DriveSource, fetch_reference
from vietmedbridge.chunks import load_bge_tokenizer, ChunkConfig
from vietmedbridge.golden import run_golden_suite

def validate_new_btc_plan(plan):
    required = "5586f423916005c11ffa8afb96895017d672af51d68aee8a88d28dd2e0dc39c6"
    if (plan.get("required_snapshot_id") != required
            or set(plan["snapshots"]) != {required}
            or not plan["sources"]
            or any(source["snapshot_id"] != required
                   or source["mode"] != "NEW_SNAPSHOT_SOURCE"
                   for source in plan["sources"])):
        raise ValueError("Lượt này chỉ nhận URL thuộc snapshot BTC mới 71be0e12; không nhập nguồn legacy.")

PLAN = read_json(CHECKOUT / "configs/intake_ai_guru_20261019.json")
validate_new_btc_plan(PLAN)
CONFIG = ChunkConfig(**PLAN["chunking"])
TOKENIZER_SPEC = {"model_id": "BAAI/bge-m3", "revision": PLAN["tokenizer_revision"], "special_tokens": False}
CONTROLLER = IntakeResume(PLAN, root=INTAKE_ROOT, work_dir=WORK_DIR,
    tokenizer_spec=TOKENIZER_SPEC, chunk_config=CONFIG, shard_size=2048)
CHECKPOINT_STATUS = CONTROLLER.status()
print("Checkpoint:", CHECKPOINT_STATUS["completed_sources"], "/", CHECKPOINT_STATUS["total_sources"],
      "nguồn đã xong | chỉ đọc receipt nhỏ, chưa tải lại dữ liệu cũ", flush=True)
LAZY_RESOURCES = {}

def load_reader():
    if "reader" not in LAZY_RESOURCES:
        from google.colab import auth
        import google.auth
        from google.auth.transport.requests import AuthorizedSession
        auth.authenticate_user()
        credentials, _ = google.auth.default()
        LAZY_RESOURCES["reader"] = DriveSource(AuthorizedSession(credentials))
    return LAZY_RESOURCES["reader"]

def load_baseline():
    if "baseline" not in LAZY_RESOURCES:
        reference = PLAN["baseline"]
        target = INTAKE_ROOT / "references/baseline_aliases.parquet"
        if not target.is_file():
            fetch_reference(load_reader(), reference, target)
        print("Cần mapping cũ để chọn phần mới; chỉ đưa mapping lên ổ local một lần.", flush=True)
        LAZY_RESOURCES["baseline"] = intake_core._local_mapping(target, reference["sha256"], WORK_DIR)
    return LAZY_RESOURCES["baseline"]

def load_links(snapshot_id):
    key = "links-"+snapshot_id
    if key not in LAZY_RESOURCES:
        spec = PLAN["snapshots"][snapshot_id]
        target = INTAKE_ROOT / "references" / (snapshot_id[:16]+"-links.parquet")
        if not target.is_file():
            print("Tải official links:", spec["revision"], flush=True)
            downloaded = Path(hf_hub_download("AIGuruTinix/ViBioMIR", spec["filename"],
                repo_type="dataset", revision=spec["revision"]))
            verify_file(downloaded, spec["sha256"])
            publish_file(downloaded, target)
        LAZY_RESOURCES[key] = intake_core._local_mapping(target, spec["sha256"], WORK_DIR)
    return LAZY_RESOURCES[key]

def load_tokenizer():
    if "tokenizer" not in LAZY_RESOURCES:
        tokenizer, spec = load_bge_tokenizer(PLAN["tokenizer_revision"])
        golden = run_golden_suite(CHECKOUT / "tests/golden/cases.json", tokenizer, spec, CONFIG)
        atomic_json(INTAKE_ROOT / "golden.json", golden)
        if not golden["passed"]:
            raise RuntimeError("Golden failed; chưa chia chunk mới.")
        LAZY_RESOURCES["tokenizer"] = (tokenizer, spec)
    return LAZY_RESOURCES["tokenizer"]


In [ ]:
#@title 3. Chỉ tiếp tục nguồn thiếu, lưu checkpoint từng shard và từng nguồn

def continue_intake():
    return CONTROLLER.run(load_reader=load_reader, load_links=load_links,
        load_baseline=load_baseline, load_tokenizer=load_tokenizer)

try:
    RESULT = continue_intake()
except OSError as error:
    if error.errno not in (103, 107):
        raise
    print("Drive ngắt kết nối; gắn lại một lần rồi tiếp tục checkpoint intake.", flush=True)
    drive.mount("/content/drive", force_remount=True)
    RESULT = continue_intake()
print(json.dumps(RESULT, ensure_ascii=False, indent=2))
print("Hoàn tất lượt nhập CPU. Giữ nguyên luồng submission đang chạy.")
print("Bàn giao lượt data mới:", INTAKE_ROOT / "intake_status.json")


`INTAKE_COMPLETE` nghĩa là mọi ID trong lượt nguồn đã có kết quả:
`ADDED_VALID_DOCUMENT`, `REUSED_EXISTING_DOCUMENT` hoặc `PIPELINE_FAILURE`.
URL lỗi được giữ trong `sources/<job>/source_outcomes.parquet` và failure
shards; trạng thái này không có nghĩa là crawl thành công 100%.

`SKIP_COMPLETED_SOURCE`: bỏ qua nguồn xong, chỉ đọc receipt/trạng thái file.
`REUSE_SELECTION`: tiếp tục nguồn dở, không chọn lại URL hoặc nạp mapping cũ.
`RESUME_PARTS`: giữ shard đã commit; chỉ source đang dở cần kiểm tra checksum.
`REUSE_VALIDATED_BUILD`: build đã qua integrity, chỉ hoàn tất bước bàn giao.
Mapping được kiểm SHA khi thực sự sử dụng; receipt không thay thế
một lần audit lại toàn bộ payload. Không nạp catalog/BM25 hoặc embedding cũ.

Gửi `intake_status.json` sau khi xong để kiểm tra số thực tế. Bước kế tiếp
là freeze/chuẩn bị index cho **lượt thêm riêng**, rồi chỉ embedding text
chưa có; ghép vào phiên submission kế tiếp sau khi ZIP hiện tại hoàn tất.
